## Modeling

#### Import thư viện

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc
)

# Advanced models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### Đọc dữ liệu

In [ ]:
df = pd.read_csv("C:\\Users\\hienm\\OneDrive\\Desktop\\Machine_Learning_2025-2026\\challange3\\data\\data_afterfix\\train_features_final_selected.csv")

TARGET = "Class"
X = df.drop(columns=[TARGET])
y = df[TARGET]

classes = sorted(y.unique())
n_classes = len(classes)

### Định nghĩa các mô hình

In [ ]:
models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000),
        {
            "C": [0.1, 1, 5, 10],
            "solver": ["liblinear", "lbfgs"]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(),
        {
            "n_estimators": [200, 300],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5]
        }
    ),

    "XGBoost": (
        XGBClassifier(n_estimators=300, eval_metric='mlogloss'),
        {
            "max_depth": [4, 6, 8],
            "learning_rate": [0.01, 0.1],
            "subsample": [0.7, 1]
        }
    ),

    "LightGBM": (
        LGBMClassifier(),
        {
            "num_leaves": [31, 60],
            "learning_rate": [0.01, 0.1],
            "n_estimators": [200, 300]
        }
    ),

    "CatBoost": (
        CatBoostClassifier(verbose=0),
        {
            "depth": [6, 8, 10],
            "learning_rate": [0.01, 0.1],
            "iterations": [200, 300]
        }
    )
}

### TRAIN + K-FOLD + GRIDSEARCHCV

In [ ]:
results = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for model_name, (model, param_grid) in models.items():
    print(f"\n=====================")
    print(f"🔍 Đang tối ưu mô hình: {model_name}")
    print("=====================")

#### Import thư viện

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc
)

# Advanced models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### Đọc dữ liệu

In [ ]:
df = pd.read_csv("C:\\Users\\hienm\\OneDrive\\Desktop\\Machine_Learning_2025-2026\\challange3\\data\\data_afterfix\\train_features_final_selected.csv")

TARGET = "Class"
X = df.drop(columns=[TARGET])
y = df[TARGET]

classes = sorted(y.unique())
n_classes = len(classes)

### Định nghĩa các mô hình

In [11]:
models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000),
        {
            "C": [0.1, 1, 5, 10],
            "solver": ["liblinear", "lbfgs"]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(),
        {
            "n_estimators": [200, 300],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5]
        }
    ),

    "XGBoost": (
        XGBClassifier(n_estimators=300, eval_metric='mlogloss'),
        {
            "max_depth": [4, 6, 8],
            "learning_rate": [0.01, 0.1],
            "subsample": [0.7, 1]
        }
    ),

    "LightGBM": (
        LGBMClassifier(),
        {
            "num_leaves": [31, 60],
            "learning_rate": [0.01, 0.1],
            "n_estimators": [200, 300]
        }
    ),

    "CatBoost": (
        CatBoostClassifier(verbose=0),
        {
            "depth": [6, 8, 10],
            "learning_rate": [0.01, 0.1],
            "iterations": [200, 300]
        }
    )
}

### TRAIN + K-FOLD + GRIDSEARCHCV

In [12]:
results = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for model_name, (model, param_grid) in models.items():
    print(f"\n=====================")
    print(f"🔍 Đang tối ưu mô hình: {model_name}")
    print("=====================")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=kf,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X, y)

    best_model = grid.best_estimator_
    best_params = grid.best_params_
    print(f"⭐ Best Params: {best_params}")

    # =============================
    # Evaluate using 5-FOLD CV
    # =============================
    acc_list = []
    prec_list = []
    rec_list = []
    f1_list = []
    
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        best_model.fit(X_train, y_train)
        y_pred = best_model.predict(X_test)

        acc_list.append(accuracy_score(y_test, y_pred))
        prec_list.append(precision_score(y_test, y_pred, average='macro'))
        rec_list.append(recall_score(y_test, y_pred, average='macro'))
        f1_list.append(f1_score(y_test, y_pred, average='macro'))

    results.append({
        "Model": model_name,
        "Accuracy": np.mean(acc_list),
        "Precision": np.mean(prec_list),
        "Recall": np.mean(rec_list),
        "F1-score": np.mean(f1_list),
        "Best Params": best_params
    })


🔍 Đang tối ưu mô hình: Logistic Regression
⭐ Best Params: {'C': 5, 'solver': 'lbfgs'}

🔍 Đang tối ưu mô hình: Random Forest
⭐ Best Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 300}

🔍 Đang tối ưu mô hình: XGBoost
⭐ Best Params: {'learning_rate': 0.1, 'max_depth': 4, 'subsample': 1}

🔍 Đang tối ưu mô hình: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001928 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5092
[LightGBM] [Info] Number of data points in the train set: 14396, number of used features: 35
[LightGBM] [Info] Start training from score -3.360098
[LightGBM] [Info] Start training from score -2.573460
[LightGBM] [Info] Start training from score -2.649110
[LightGBM] [Info] Start training from score -3.800154
[LightGBM] [Info] Start training from score -3.838133
[LightGBM] [Info] Start training from score -2.521120
[LightGBM] [Info] Start training from score -1.

### Tỏng kết


In [13]:
results_df = pd.DataFrame(results)
print("\n\n=====================")
print("🏆 KẾT QUẢ TỔNG HỢP 5 MÔ HÌNH")
print("=====================")
print(results_df.sort_values(by="F1-score", ascending=False))



🏆 KẾT QUẢ TỔNG HỢP 5 MÔ HÌNH
                 Model  Accuracy  Precision    Recall  F1-score  \
2              XGBoost  0.627952   0.699759  0.661480  0.668983   
3             LightGBM  0.628230   0.706736  0.653947  0.664279   
4             CatBoost  0.626146   0.692939  0.651041  0.654449   
1        Random Forest  0.591484   0.659914  0.625244  0.634570   
0  Logistic Regression  0.506252   0.516725  0.528612  0.518835   

                                         Best Params  
2  {'learning_rate': 0.1, 'max_depth': 4, 'subsam...  
3  {'learning_rate': 0.01, 'n_estimators': 300, '...  
4  {'depth': 6, 'iterations': 300, 'learning_rate...  
1  {'max_depth': None, 'min_samples_split': 5, 'n...  
0                        {'C': 5, 'solver': 'lbfgs'}  
